# Dataset Download

This notebook downloads and organizes the fall detection datasets.

## Datasets:
1. **UR Fall Detection Dataset** - Universidad de Rzeszów, Poland
   - 70 sequences (30 falls, 40 ADL - Activities of Daily Living)
   - Resolution: 640×480
   - FPS: 30
   - Source: http://fenix.ur.edu.pl/~mkepski/ds/uf.html

2. **Le2i Fall Detection Dataset** - Université de Bourgogne, France
   - ~200 videos in varied scenarios (home, office, conference room)
   - Multiple environments and angles
   - Source: https://imvia.u-bourgogne.fr/en/database/fall-detection-dataset-2.html

## Goals:
1. Download UR Fall Detection dataset
2. Download Le2i Fall Detection dataset
3. Organize files into proper directory structure
4. Verify downloads and create metadata
5. Save to Google Drive (if running in Colab)

## 0. Setup Environment

In [1]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally")

✓ Running in Google Colab


In [ ]:
# Standard imports
import os
import sys
import requests
import zipfile
import tarfile
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import shutil
from bs4 import BeautifulSoup
from typing import List, Dict
import time
import subprocess

## 1. Configuration

In [3]:
# Configuration: Google Drive folder name
# Change this if you want to use a different folder name in Google Drive
GOOGLE_DRIVE_PROJECT_FOLDER = 'safeguard-vision-ai'

print(f"✓ Project folder configured: {GOOGLE_DRIVE_PROJECT_FOLDER}")

✓ Project folder configured: safeguard-vision-ai


## 2. Setup Google Drive Integration (For Colab)

In [4]:
if IN_COLAB:
    print("=" * 60)
    print("Google Colab Environment Detected")
    print("=" * 60)

    # Initialize Drive mount flag
    USE_DRIVE = False

    # Try to mount Google Drive
    from google.colab import drive

    # Check if already mounted
    if os.path.exists('/content/drive/MyDrive'):
        print("✓ Google Drive already mounted")
        USE_DRIVE = True
    else:
        print("\n⚠ Attempting to mount Google Drive...")
        print("Note: This may fail in VS Code (that's expected)")

        try:
            drive.mount('/content/drive', force_remount=False)
            print("✓ Google Drive mounted successfully!")
            USE_DRIVE = True
        except ValueError as e:
            print(f"\n✗ Drive mount failed: {str(e)[:50]}...")
            print("\n" + "=" * 60)
            print("VS CODE WORKAROUND - Using Local Storage")
            print("=" * 60)
            print("Files will be saved to: /content/safeguard-vision-ai/data/raw/")
            print("⚠ WARNING: Files are temporary and will be lost on disconnect!")
            print("\nTo preserve files:")
            print("  1. Download them before disconnecting")
            print("  2. Or use Colab in browser: https://colab.research.google.com")
            print("=" * 60)
            USE_DRIVE = False
        except Exception as e:
            print(f"\n✗ Unexpected error: {type(e).__name__}")
            print("Continuing with local storage...")
            USE_DRIVE = False

    # Clone repository
    if not os.path.exists('/content/safeguard-vision-ai'):
        print("\nCloning repository...")
        !git clone https://github.com/hugoangeles0810/safeguard-vision-ai.git
        os.chdir('/content/safeguard-vision-ai')

        print("Installing dependencies...")
        !pip install -q -r requirements.txt
        print("✓ Dependencies installed")
    else:
        print("\n✓ Repository already exists")
        os.chdir('/content/safeguard-vision-ai')

    print(f"Current directory: {os.getcwd()}")

    # Add src to Python path
    repo_src_path = '/content/safeguard-vision-ai/src'
    if repo_src_path not in sys.path:
        sys.path.insert(0, repo_src_path)
        print(f"✓ Added to path: {repo_src_path}")

    # Configure paths based on Drive availability
    if USE_DRIVE:
        print("\n" + "=" * 60)
        print("Configuring Google Drive Storage")
        print("=" * 60)
        try:
            from utils.drive_utils import setup_paths
            paths = setup_paths(project_name=GOOGLE_DRIVE_PROJECT_FOLDER)
            download_dir = Path('/content/dataset_downloads')
            download_dir.mkdir(exist_ok=True)

            print(f"✓ Project folder: {GOOGLE_DRIVE_PROJECT_FOLDER}")
            print(f"✓ Temp downloads: {download_dir}")
            print(f"✓ Final destination: {paths['data_raw']}")
        except Exception as e:
            print(f"✗ Error setting up Drive paths: {e}")
            print("Falling back to local storage...")
            USE_DRIVE = False

    if not USE_DRIVE:
        print("\n" + "=" * 60)
        print("Configuring Local Storage")
        print("=" * 60)
        local_data_dir = Path('/content/safeguard-vision-ai/data/raw')
        local_data_dir.mkdir(parents=True, exist_ok=True)

        paths = {
            'project_root': Path('/content/safeguard-vision-ai'),
            'data_raw': local_data_dir,
        }
        download_dir = local_data_dir

        print(f"✓ Storage location: {paths['data_raw']}")
        print("✓ Files are accessible within this session")

else:
    # Running locally (not in Colab)
    print("=" * 60)
    print("Local Environment Detected")
    print("=" * 60)

    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    paths = {
        'project_root': project_root,
        'data_raw': project_root / 'data' / 'raw',
    }
    download_dir = project_root / 'data' / 'downloads'
    download_dir.mkdir(parents=True, exist_ok=True)

    print(f"✓ Download location: {download_dir}")
    print(f"✓ Destination: {paths['data_raw']}")

# Final summary
print(f"\n{'='*60}")
print("CONFIGURATION COMPLETE")
print(f"{'='*60}")
print(f"✓ Data destination: {paths['data_raw']}")
print(f"✓ Download directory: {download_dir}")
print(f"{'='*60}")

Google Colab Environment Detected

⚠ Attempting to mount Google Drive...
Note: This may fail in VS Code (that's expected)
Mounted at /content/drive
✓ Google Drive mounted successfully!

Cloning repository...
Cloning into 'safeguard-vision-ai'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 83 (delta 26), reused 78 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 61.66 KiB | 2.28 MiB/s, done.
Resolving deltas: 100% (26/26), done.
Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.0/118.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

## 3. Helper Functions

In [ ]:
def download_file(url: str, destination: Path, description: str = "Downloading") -> Path:
    """
    Download a file from a URL with progress bar.

    Args:
        url: URL to download from
        destination: Path where to save the file
        description: Description for progress bar

    Returns:
        Path to downloaded file
    """
    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()

        total_size = int(response.headers.get('content-length', 0))

        destination.parent.mkdir(parents=True, exist_ok=True)

        with open(destination, 'wb') as file:
            with tqdm(total=total_size, unit='B', unit_scale=True, desc=description) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
                        pbar.update(len(chunk))

        print(f"✓ Downloaded: {destination.name}")
        return destination
    except Exception as e:
        print(f"✗ Error downloading {url}: {e}")
        if destination.exists():
            destination.unlink()
        return None


def extract_archive(archive_path: Path, extract_to: Path) -> Path:
    """
    Extract a zip or tar archive.

    Args:
        archive_path: Path to the archive file
        extract_to: Directory where to extract

    Returns:
        Path to extraction directory
    """
    extract_to.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {archive_path.name}...")

    if archive_path.suffix == '.zip':
        with zipfile.ZipFile(archive_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
    elif archive_path.suffix in ['.tar', '.gz', '.tgz']:
        with tarfile.open(archive_path, 'r:*') as tar_ref:
            tar_ref.extractall(extract_to)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path.suffix}")

    print(f"✓ Extracted to: {extract_to}")
    return extract_to


def count_video_files(directory: Path) -> dict:
    """
    Count video files in a directory.

    Args:
        directory: Directory to search

    Returns:
        Dictionary with counts by extension
    """
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.mpeg', '.mpg']
    counts = {}
    total = 0

    for ext in video_extensions:
        files = list(directory.rglob(f'*{ext}'))
        if files:
            counts[ext] = len(files)
            total += len(files)

    counts['total'] = total
    return counts


def is_image_sequence_folder(folder: Path) -> bool:
    """
    Check if a folder contains an image sequence.

    Args:
        folder: Path to check

    Returns:
        True if folder contains image files (jpg, png, etc.)
    """
    if not folder.is_dir():
        return False

    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']

    # Check if folder contains at least 5 image files
    image_count = 0
    for ext in image_extensions:
        image_count += len(list(folder.glob(f'*{ext}')))
        if image_count >= 5:  # Minimum threshold for a sequence
            return True

    return False


def count_image_sequences(directory: Path, recursive: bool = True) -> dict:
    """
    Count image sequence folders in a directory.

    Args:
        directory: Directory to search
        recursive: Whether to search recursively

    Returns:
        Dictionary with counts and list of sequence folders
    """
    if not directory.exists():
        return {'total': 0, 'sequences': []}

    sequences = []

    if recursive:
        # Find all subdirectories that contain images
        for item in directory.rglob('*'):
            if item.is_dir() and is_image_sequence_folder(item):
                sequences.append(item)
    else:
        # Only check immediate subdirectories
        for item in directory.iterdir():
            if item.is_dir() and is_image_sequence_folder(item):
                sequences.append(item)

    return {
        'total': len(sequences),
        'sequences': sequences
    }


def count_videos_or_sequences(directory: Path) -> dict:
    """
    Count both video files and image sequence folders.

    Args:
        directory: Directory to search

    Returns:
        Dictionary with total count
    """
    if not directory.exists():
        return {'total': 0}

    # Count video files
    video_count = count_video_files(directory)

    # Count image sequences
    sequence_count = count_image_sequences(directory)

    total = video_count['total'] + sequence_count['total']

    return {
        'video_files': video_count['total'],
        'image_sequences': sequence_count['total'],
        'total': total
    }


def generate_ur_fall_urls() -> Dict[str, List[str]]:
    """
    Generate download URLs for UR Fall Detection dataset.

    Returns:
        Dictionary with 'fall' and 'adl' keys containing lists of video URLs
    """
    base_url = "https://fenix.ur.edu.pl/mkepski/ds/data/"

    urls = {
        'fall': [],
        'adl': []
    }

    # Fall sequences: 30 sequences, 2 cameras each (cam0, cam1)
    for i in range(1, 31):
        for cam in [0, 1]:
            video_name = f"fall-{i:02d}-cam{cam}.mp4"
            urls['fall'].append(base_url + video_name)

    # ADL sequences: 40 sequences, 1 camera only (cam0)
    for i in range(1, 41):
        video_name = f"adl-{i:02d}-cam0.mp4"
        urls['adl'].append(base_url + video_name)

    return urls


def download_ur_fall_dataset(urls: Dict[str, List[str]], destination: Path,
                             download_temp: Path = None) -> Dict[str, int]:
    """
    Download UR Fall Detection dataset videos.

    Args:
        urls: Dictionary with 'fall' and 'adl' video URLs
        destination: Final destination directory (e.g., Google Drive)
        download_temp: Temporary download location (for Colab)

    Returns:
        Dictionary with download statistics
    """
    if download_temp is None:
        download_temp = destination

    stats = {
        'fall_downloaded': 0,
        'adl_downloaded': 0,
        'fall_failed': 0,
        'adl_failed': 0,
    }

    # Create directories
    fall_temp_dir = download_temp / 'ur_fall' / 'fall'
    adl_temp_dir = download_temp / 'ur_fall' / 'adl'
    fall_temp_dir.mkdir(parents=True, exist_ok=True)
    adl_temp_dir.mkdir(parents=True, exist_ok=True)

    # Download fall sequences
    print("\n" + "=" * 60)
    print("Downloading Fall Sequences")
    print("=" * 60)
    for i, url in enumerate(urls['fall'], 1):
        filename = url.split('/')[-1]
        dest_path = fall_temp_dir / filename

        # Skip if already exists
        if dest_path.exists():
            print(f"[{i}/{len(urls['fall'])}] Skipping {filename} (already exists)")
            stats['fall_downloaded'] += 1
            continue

        print(f"\n[{i}/{len(urls['fall'])}] Downloading {filename}...")
        result = download_file(url, dest_path, description=filename)

        if result:
            stats['fall_downloaded'] += 1
        else:
            stats['fall_failed'] += 1

        # Small delay to avoid overwhelming the server
        time.sleep(0.5)

    # Download ADL sequences
    print("\n" + "=" * 60)
    print("Downloading ADL Sequences")
    print("=" * 60)
    for i, url in enumerate(urls['adl'], 1):
        filename = url.split('/')[-1]
        dest_path = adl_temp_dir / filename

        # Skip if already exists
        if dest_path.exists():
            print(f"[{i}/{len(urls['adl'])}] Skipping {filename} (already exists)")
            stats['adl_downloaded'] += 1
            continue

        print(f"\n[{i}/{len(urls['adl'])}] Downloading {filename}...")
        result = download_file(url, dest_path, description=filename)

        if result:
            stats['adl_downloaded'] += 1
        else:
            stats['adl_failed'] += 1

        # Small delay to avoid overwhelming the server
        time.sleep(0.5)

    # Move to final destination if using temporary location
    if download_temp != destination:
        print("\n" + "=" * 60)
        print("Moving files to Google Drive...")
        print("=" * 60)

        final_fall_dir = destination / 'ur_fall' / 'fall'
        final_adl_dir = destination / 'ur_fall' / 'adl'
        final_fall_dir.mkdir(parents=True, exist_ok=True)
        final_adl_dir.mkdir(parents=True, exist_ok=True)

        # Move fall videos
        for video in fall_temp_dir.glob('*.mp4'):
            shutil.move(str(video), str(final_fall_dir / video.name))
            print(f"✓ Moved {video.name} to Drive")

        # Move ADL videos
        for video in adl_temp_dir.glob('*.mp4'):
            shutil.move(str(video), str(final_adl_dir / video.name))
            print(f"✓ Moved {video.name} to Drive")

        print("\n✓ All files moved to Google Drive")

    return stats


def download_le2i_dataset(google_drive_id: str, destination: Path,
                          download_temp: Path = None) -> Dict[str, any]:
    """
    Download Le2i Fall Detection dataset from Google Drive.

    Le2i dataset structure after extraction:
    - Contains image sequences (folders with images) instead of video files
    - 2025/ or other year folders containing:
        - Fall/          -> contains fall image sequences
        - Blank/         -> ADL activities (blank/empty scenes)
        - Lie/           -> ADL activities (lying down)
        - Likefall/      -> ADL activities (fall-like movements)
        - Stand/         -> ADL activities (standing)

    Each category folder contains subfolders representing individual sequences.
    Each sequence folder contains a series of images (jpg, png, etc.).

    Args:
        google_drive_id: Google Drive file ID
        destination: Final destination directory (e.g., Google Drive)
        download_temp: Temporary download location (for Colab)

    Returns:
        Dictionary with download statistics
    """
    if download_temp is None:
        download_temp = destination

    stats = {
        'downloaded': False,
        'extracted': False,
        'fall_moved': 0,
        'adl_moved': 0,
        'fall_failed': 0,
        'adl_failed': 0,
    }

    try:
        # Install gdown if not available
        try:
            import gdown
        except ImportError:
            print("Installing gdown...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            import gdown

        # Create temporary directory for download
        temp_dir = download_temp / 'le2i_temp'
        temp_dir.mkdir(parents=True, exist_ok=True)

        # Download from Google Drive
        print("\n" + "=" * 60)
        print("Downloading Le2i Dataset from Google Drive")
        print("=" * 60)

        # Construct download URL
        url = f"https://drive.google.com/uc?id={google_drive_id}"
        archive_path = temp_dir / "le2i_dataset.zip"

        print(f"Downloading from Google Drive (ID: {google_drive_id})...")
        print(f"This may take several minutes...")

        try:
            gdown.download(url, str(archive_path), quiet=False, fuzzy=True)
            stats['downloaded'] = True
            print(f"✓ Downloaded to: {archive_path}")
        except Exception as e:
            print(f"✗ Download failed: {e}")
            print("\nTrying alternative download method...")
            try:
                gdown.download(f"https://drive.google.com/uc?export=download&id={google_drive_id}",
                             str(archive_path), quiet=False)
                stats['downloaded'] = True
                print(f"✓ Downloaded to: {archive_path}")
            except Exception as e2:
                print(f"✗ Alternative download also failed: {e2}")
                print("\n⚠ Manual download required:")
                print(f"   1. Visit: https://drive.google.com/file/d/{google_drive_id}/view")
                print(f"   2. Download the file manually")
                print(f"   3. Upload to: {temp_dir}")
                return stats

        # Extract archive
        print("\n" + "=" * 60)
        print("Extracting Le2i Dataset")
        print("=" * 60)

        extract_dir = temp_dir / 'extracted'
        try:
            extract_archive(archive_path, extract_dir)
            stats['extracted'] = True
        except Exception as e:
            print(f"✗ Extraction failed: {e}")
            return stats

        # Organize image sequences into fall/adl directories
        print("\n" + "=" * 60)
        print("Organizing Le2i Image Sequences")
        print("=" * 60)
        print("Note: Le2i dataset contains image sequences (folders), not video files")

        # Create temporary fall/adl directories
        fall_temp_dir = download_temp / 'le2i' / 'fall'
        adl_temp_dir = download_temp / 'le2i' / 'adl'
        fall_temp_dir.mkdir(parents=True, exist_ok=True)
        adl_temp_dir.mkdir(parents=True, exist_ok=True)

        # Le2i dataset structure:
        # Fall image sequences are in "Fall" folder
        # ADL image sequences are in "Blank", "Lie", "Likefall", "Stand" folders
        fall_folders = ['Fall']
        adl_folders = ['Blank', 'Lie', 'Likefall', 'Stand']

        # Process Fall image sequences
        print("\nProcessing Fall image sequences...")
        for folder_name in fall_folders:
            # Search for the folder in extracted directory
            fall_category_dirs = list(extract_dir.rglob(folder_name))
            for fall_category_dir in fall_category_dirs:
                if fall_category_dir.is_dir():
                    print(f"Found Fall category folder: {fall_category_dir}")

                    # Find all image sequence folders within this category
                    for item in fall_category_dir.iterdir():
                        if item.is_dir() and is_image_sequence_folder(item):
                            try:
                                # Copy the entire sequence folder
                                dest_folder = fall_temp_dir / item.name
                                if dest_folder.exists():
                                    # Add a unique suffix if folder already exists
                                    counter = 1
                                    while (fall_temp_dir / f"{item.name}_{counter}").exists():
                                        counter += 1
                                    dest_folder = fall_temp_dir / f"{item.name}_{counter}"

                                shutil.copytree(item, dest_folder)
                                stats['fall_moved'] += 1
                                print(f"✓ Fall sequence: {item.name}")
                            except Exception as e:
                                print(f"✗ Failed to copy {item.name}: {e}")
                                stats['fall_failed'] += 1

        # Process ADL image sequences
        print("\nProcessing ADL image sequences...")
        for folder_name in adl_folders:
            # Search for the folder in extracted directory
            adl_category_dirs = list(extract_dir.rglob(folder_name))
            for adl_category_dir in adl_category_dirs:
                if adl_category_dir.is_dir():
                    print(f"Found ADL category folder: {adl_category_dir}")

                    # Find all image sequence folders within this category
                    for item in adl_category_dir.iterdir():
                        if item.is_dir() and is_image_sequence_folder(item):
                            try:
                                # Add category prefix to avoid name conflicts
                                dest_folder_name = f"{folder_name}_{item.name}"
                                dest_folder = adl_temp_dir / dest_folder_name

                                if dest_folder.exists():
                                    # Add a unique suffix if folder already exists
                                    counter = 1
                                    while (adl_temp_dir / f"{dest_folder_name}_{counter}").exists():
                                        counter += 1
                                    dest_folder = adl_temp_dir / f"{dest_folder_name}_{counter}"

                                shutil.copytree(item, dest_folder)
                                stats['adl_moved'] += 1
                                print(f"✓ ADL sequence ({folder_name}): {item.name}")
                            except Exception as e:
                                print(f"✗ Failed to copy {item.name}: {e}")
                                stats['adl_failed'] += 1

        # Summary
        print(f"\n✓ Organized {stats['fall_moved']} fall image sequences")
        print(f"✓ Organized {stats['adl_moved']} ADL image sequences")

        # Move to final destination if using temporary location
        if download_temp != destination:
            print("\n" + "=" * 60)
            print("Moving files to Google Drive...")
            print("=" * 60)

            final_fall_dir = destination / 'le2i' / 'fall'
            final_adl_dir = destination / 'le2i' / 'adl'
            final_fall_dir.mkdir(parents=True, exist_ok=True)
            final_adl_dir.mkdir(parents=True, exist_ok=True)

            # Move fall image sequences
            moved_fall = 0
            for sequence_folder in fall_temp_dir.iterdir():
                if sequence_folder.is_dir():
                    dest = final_fall_dir / sequence_folder.name
                    if dest.exists():
                        shutil.rmtree(dest)  # Remove existing if present
                    shutil.move(str(sequence_folder), str(dest))
                    moved_fall += 1
            print(f"✓ Moved {moved_fall} fall image sequences to Drive")

            # Move ADL image sequences
            moved_adl = 0
            for sequence_folder in adl_temp_dir.iterdir():
                if sequence_folder.is_dir():
                    dest = final_adl_dir / sequence_folder.name
                    if dest.exists():
                        shutil.rmtree(dest)  # Remove existing if present
                    shutil.move(str(sequence_folder), str(dest))
                    moved_adl += 1
            print(f"✓ Moved {moved_adl} ADL image sequences to Drive")

            print("\n✓ All files moved to Google Drive")

        # Cleanup temporary files
        print("\n" + "=" * 60)
        print("Cleaning up temporary files...")
        print("=" * 60)
        try:
            shutil.rmtree(temp_dir)
            print("✓ Temporary files cleaned up")
        except Exception as e:
            print(f"⚠ Could not clean up temp files: {e}")

    except Exception as e:
        print(f"✗ Error during Le2i download: {e}")
        import traceback
        traceback.print_exc()

    return stats

## 4. Download UR Fall Detection Dataset

**Dataset Information:**
- 70 sequences total
- 30 fall sequences
- 40 ADL (Activities of Daily Living) sequences
- RGB + Depth data available
- Resolution: 640×480 @ 30fps

In [6]:
# UR Fall Dataset configuration
ur_fall_base_url = "https://fenix.ur.edu.pl/mkepski/ds/data/"
ur_fall_destination = paths['data_raw'] / 'ur_fall'

print("=" * 60)
print("UR Fall Detection Dataset Download")
print("=" * 60)
print(f"\nBase URL: {ur_fall_base_url}")
print(f"Destination: {ur_fall_destination}")
print("\nDataset structure:")
print("  - 30 fall sequences (60 videos: 2 cameras per sequence)")
print("  - 40 ADL sequences (40 videos: 1 camera per sequence)")
print("  - Total: 100 video files")
print()

UR Fall Detection Dataset Download

Base URL: https://fenix.ur.edu.pl/mkepski/ds/data/
Destination: /content/drive/MyDrive/safeguard-vision-ai/data/raw/ur_fall

Dataset structure:
  - 30 fall sequences (60 videos: 2 cameras per sequence)
  - 40 ADL sequences (40 videos: 1 camera per sequence)
  - Total: 100 video files



In [7]:
# Generate download URLs
print("Generating download URLs...")
ur_fall_urls = generate_ur_fall_urls()

print(f"\n✓ Generated {len(ur_fall_urls['fall'])} fall video URLs")
print(f"✓ Generated {len(ur_fall_urls['adl'])} ADL video URLs")
print(f"  Total: {len(ur_fall_urls['fall']) + len(ur_fall_urls['adl'])} videos")

# Preview URLs
print("\nSample URLs:")
print(f"  Fall: {ur_fall_urls['fall'][0]}")
print(f"  ADL:  {ur_fall_urls['adl'][0]}")

Generating download URLs...

✓ Generated 60 fall video URLs
✓ Generated 40 ADL video URLs
  Total: 100 videos

Sample URLs:
  Fall: https://fenix.ur.edu.pl/mkepski/ds/data/fall-01-cam0.mp4
  ADL:  https://fenix.ur.edu.pl/mkepski/ds/data/adl-01-cam0.mp4


In [ ]:
# Ready to download
print("=" * 60)
print("Ready to Download UR Fall Dataset")
print("=" * 60)
print("\nThis will download ~100 video files.")
print("Estimated time: 15-30 minutes depending on connection speed.")
print("\nNote: Downloads will resume if interrupted.")
print(f"\nFiles will be downloaded to: {download_dir}")
print("\n⚠ Set START_DOWNLOAD = True in the cell above to begin.")

In [9]:
# Execute download
# Set to True to start download, False to skip
START_DOWNLOAD = True

if START_DOWNLOAD:
    if IN_COLAB:
        # Download to temporary location, then move to Drive
        download_stats = download_ur_fall_dataset(
            ur_fall_urls,
            destination=paths['data_raw'],
            download_temp=download_dir
        )
    else:
        # Download directly to destination
        download_stats = download_ur_fall_dataset(
            ur_fall_urls,
            destination=paths['data_raw']
        )

    # Print summary
    print("\n" + "=" * 60)
    print("DOWNLOAD COMPLETE")
    print("=" * 60)
    print(f"\nFall videos downloaded: {download_stats['fall_downloaded']}")
    print(f"Fall videos failed: {download_stats['fall_failed']}")
    print(f"ADL videos downloaded: {download_stats['adl_downloaded']}")
    print(f"ADL videos failed: {download_stats['adl_failed']}")

    total_downloaded = download_stats['fall_downloaded'] + download_stats['adl_downloaded']
    total_failed = download_stats['fall_failed'] + download_stats['adl_failed']

    print(f"\nTotal downloaded: {total_downloaded}")
    print(f"Total failed: {total_failed}")

    if total_failed == 0:
        print("\n✓ All videos downloaded successfully!")
    else:
        print(f"\n⚠ {total_failed} videos failed to download. You may need to retry.")
else:
    print("\n⚠ Download skipped. Set START_DOWNLOAD = True to begin download.")



[1/60] Downloading fall-01-cam0.mp4...


fall-01-cam0.mp4: 100%|██████████| 1.30M/1.30M [00:00<00:00, 1.61MB/s]


✓ Downloaded: fall-01-cam0.mp4

[2/60] Downloading fall-01-cam1.mp4...


fall-01-cam1.mp4: 100%|██████████| 1.32M/1.32M [00:00<00:00, 1.66MB/s]


✓ Downloaded: fall-01-cam1.mp4

[3/60] Downloading fall-02-cam0.mp4...


...

✓ Downloaded: adl-38-cam0.mp4

[39/40] Downloading adl-39-cam0.mp4...


adl-39-cam0.mp4: 100%|██████████| 2.24M/2.24M [00:00<00:00, 2.42MB/s]


✓ Downloaded: adl-39-cam0.mp4

[40/40] Downloading adl-40-cam0.mp4...


adl-40-cam0.mp4: 100%|██████████| 2.78M/2.78M [00:00<00:00, 3.00MB/s]


✓ Downloaded: adl-40-cam0.mp4

Moving files to Google Drive...
✓ Moved fall-06-cam1.mp4 to Drive
✓ Moved fall-03-cam1.mp4 to Drive
✓ Moved fall-12-cam1.mp4 to Drive
✓ Moved fall-18-cam1.mp4 to Drive
✓ Moved fall-14-cam1.mp4 to Drive
✓ Moved fall-04-cam0.mp4 to Drive
✓ Moved fall-05-cam1.mp4 to Drive
✓ Moved fall-12-cam0.mp4 to Drive
✓ Moved fall-08-cam0.mp4 to Drive
✓ Moved fall-25-cam1.mp4 to Drive
✓ Moved fall-16-cam1.mp4 to Drive
✓ Moved fall-26-cam1.mp4 to Drive
✓ Moved fall-21-cam1.mp4 to Drive
✓ Moved fall-30-cam1.mp4 to Drive
✓ Moved fall-29-cam0.mp4 to Drive
✓ Moved fall-22-cam0.mp4 to Drive
✓ Moved fall-07-cam0.mp4 to Drive
✓ Moved fall-14-cam0.mp4 to Drive
✓ Moved fall-22-cam1.mp4 to Drive
✓ Moved fall-17-cam1.mp4 to Drive
✓ Moved fall-03-cam0.mp4 to Drive
✓ Moved fall-13-cam1.mp4 to Drive
✓ Moved fall-15-cam1.mp4 to Drive
✓ Moved fall-11-cam1.mp4 to Drive
✓ Moved fall-20-cam0.mp4 to Drive
✓ Moved fall-04-cam1.mp4 to Drive
✓ Moved fall-17-cam0.mp4 to Drive
✓ Moved fall-20-cam

## 5. Download Le2i Fall Detection Dataset

**Dataset Information:**
- ~200 image sequences (NOT video files)
- Each sequence is a folder containing a series of images (jpg, png, etc.)
- Multiple camera angles and environments
- Includes falls and ADL activities
- Various resolutions and frame rates

**Important Note:** Le2i dataset contains image sequences in folders, not traditional video files. The notebook will automatically handle this format.

In [ ]:
# Le2i Dataset URLs
# Visit: https://imvia.u-bourgogne.fr/en/database/fall-detection-dataset-2.html

le2i_url = "https://drive.google.com/file/d/16sv5CLT3pBI0kcLvAYJqT_zscGK-GF91/view?usp=sharing"  # Example URL - verify on website
le2i_destination = paths['data_raw'] / 'le2i'

print("=" * 60)
print("Le2i Fall Detection Dataset Download")
print("=" * 60)
print("\nDataset source: https://imvia.u-bourgogne.fr/en/database/fall-detection-dataset-2.html")
print("\nThis dataset may also require registration or manual download.")
if IN_COLAB:
    print(f"Upload to: MyDrive/{GOOGLE_DRIVE_PROJECT_FOLDER}/data/raw/le2i/")
else:
    print(f"Or place in: {le2i_destination}")
print()

In [ ]:
# Le2i Dataset Google Drive URL
# Update this URL if the dataset location changes
# Format: https://drive.google.com/file/d/FILE_ID/view?usp=sharing
# Or just provide the FILE_ID directly
le2i_google_drive_url = "16sv5CLT3pBI0kcLvAYJqT_zscGK-GF91"  # File ID from the sharing URL
le2i_destination = paths['data_raw'] / 'le2i'

print("=" * 60)
print("Le2i Fall Detection Dataset Download")
print("=" * 60)
print(f"\nGoogle Drive File ID: {le2i_google_drive_url}")
print(f"Destination: {le2i_destination}")
print("\nDataset information:")
print("  - ~200 videos in varied scenarios")
print("  - Multiple camera angles and environments")
print("  - Includes falls and ADL activities")
print("  - Various resolutions and frame rates")
print("\nNote: The dataset will be downloaded from Google Drive,")
print("      extracted, and organized into fall/adl subdirectories.")
print()

# Execute download
# Set to True to start download, False to skip
START_LE2I_DOWNLOAD = True

if START_LE2I_DOWNLOAD:
    if IN_COLAB:
        # Download to temporary location, then move to Drive
        le2i_stats = download_le2i_dataset(
            le2i_google_drive_url,
            destination=paths['data_raw'],
            download_temp=download_dir
        )
    else:
        # Download directly to destination
        le2i_stats = download_le2i_dataset(
            le2i_google_drive_url,
            destination=paths['data_raw']
        )

    # Print summary
    print("\n" + "=" * 60)
    print("LE2I DOWNLOAD COMPLETE")
    print("=" * 60)
    print(f"\nDownloaded: {le2i_stats.get('downloaded', False)}")
    print(f"Extracted: {le2i_stats.get('extracted', False)}")
    print(f"Fall videos organized: {le2i_stats.get('fall_moved', 0)}")
    print(f"ADL videos organized: {le2i_stats.get('adl_moved', 0)}")
    print(f"Fall videos failed: {le2i_stats.get('fall_failed', 0)}")
    print(f"ADL videos failed: {le2i_stats.get('adl_failed', 0)}")

    total_organized = le2i_stats.get('fall_moved', 0) + le2i_stats.get('adl_moved', 0)
    total_failed = le2i_stats.get('fall_failed', 0) + le2i_stats.get('adl_failed', 0)

    print(f"\nTotal videos organized: {total_organized}")
    print(f"Total failed: {total_failed}")

    if total_organized > 0:
        print(f"\n✓ Le2i dataset successfully downloaded and organized!")
        print(f"  Location: {le2i_destination}")
    else:
        print(f"\n⚠ No videos were organized. Please check the dataset structure.")
        print(f"  You may need to manually organize videos in: {le2i_destination}")
else:
    print("\n⚠ Download skipped. Set START_LE2I_DOWNLOAD = True to begin download.")

## 6. Organize Dataset Structure

Ensure datasets are organized in the expected structure:

```
data/raw/
├── ur_fall/
│   ├── fall/
│   │   ├── video_001.avi
│   │   └── ...
│   └── adl/
│       ├── video_001.avi
│       └── ...
└── le2i/
    ├── fall/
    │   └── ...
    └── adl/
        └── ...
```

In [ ]:
def organize_dataset(dataset_path: Path, dataset_name: str):
    """
    Organize dataset into fall/adl subdirectories if not already organized.

    Args:
        dataset_path: Path to dataset directory
        dataset_name: Name of the dataset (for logging)
    """
    fall_dir = dataset_path / 'fall'
    adl_dir = dataset_path / 'adl'

    fall_dir.mkdir(parents=True, exist_ok=True)
    adl_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nOrganizing {dataset_name} dataset...")
    print(f"  Fall directory: {fall_dir}")
    print(f"  ADL directory: {adl_dir}")

    # TODO: Implement organization logic based on actual dataset structure
    # This will depend on how the datasets are originally organized

    print(f"✓ {dataset_name} dataset organized")


# Organize both datasets
# organize_dataset(paths['data_raw'] / 'ur_fall', 'UR Fall')
# organize_dataset(paths['data_raw'] / 'le2i', 'Le2i')

print("⚠ Dataset organization logic needs to be implemented")

## 7. Verify Downloads and Create Metadata

In [ ]:
def verify_dataset(dataset_path: Path, dataset_name: str) -> dict:
    """
    Verify dataset download and count files/sequences.

    Args:
        dataset_path: Path to dataset directory
        dataset_name: Name of the dataset

    Returns:
        Dictionary with verification results
    """
    print(f"\nVerifying {dataset_name} dataset...")
    print(f"Location: {dataset_path}")

    if not dataset_path.exists():
        print(f"✗ Directory does not exist: {dataset_path}")
        return {'exists': False}

    fall_dir = dataset_path / 'fall'
    adl_dir = dataset_path / 'adl'

    # Count both video files and image sequences
    fall_count = count_videos_or_sequences(fall_dir) if fall_dir.exists() else {'total': 0}
    adl_count = count_videos_or_sequences(adl_dir) if adl_dir.exists() else {'total': 0}

    results = {
        'exists': True,
        'dataset': dataset_name,
        'path': str(dataset_path),
        'fall_videos': fall_count.get('video_files', 0),
        'fall_sequences': fall_count.get('image_sequences', 0),
        'fall_total': fall_count.get('total', 0),
        'adl_videos': adl_count.get('video_files', 0),
        'adl_sequences': adl_count.get('image_sequences', 0),
        'adl_total': adl_count.get('total', 0),
        'total_items': fall_count.get('total', 0) + adl_count.get('total', 0),
    }

    print(f"  Fall videos: {results['fall_videos']}")
    print(f"  Fall image sequences: {results['fall_sequences']}")
    print(f"  Fall total: {results['fall_total']}")
    print(f"  ADL videos: {results['adl_videos']}")
    print(f"  ADL image sequences: {results['adl_sequences']}")
    print(f"  ADL total: {results['adl_total']}")
    print(f"  Total items: {results['total_items']}")

    if results['total_items'] > 0:
        print(f"✓ {dataset_name} dataset verified")
    else:
        print(f"⚠ No videos or sequences found in {dataset_name} dataset")

    return results

In [ ]:
# Verify all datasets
print("=" * 60)
print("Dataset Verification")
print("=" * 60)

ur_fall_info = verify_dataset(paths['data_raw'] / 'ur_fall', 'UR Fall')
le2i_info = verify_dataset(paths['data_raw'] / 'le2i', 'Le2i')

In [ ]:
# Create summary metadata
metadata = [
    ur_fall_info,
    le2i_info
]

df_metadata = pd.DataFrame(metadata)
print("\n" + "=" * 60)
print("Dataset Summary")
print("=" * 60)
print(df_metadata.to_string(index=False))

# Save metadata
metadata_path = paths['data_raw'] / 'dataset_metadata.csv'
df_metadata.to_csv(metadata_path, index=False)
print(f"\n✓ Metadata saved to: {metadata_path}")

## 8. Summary and Next Steps

In [ ]:
total_fall_items = ur_fall_info.get('fall_total', 0) + le2i_info.get('fall_total', 0)
total_adl_items = ur_fall_info.get('adl_total', 0) + le2i_info.get('adl_total', 0)
total_items = total_fall_items + total_adl_items

print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
print("=" * 60)
print(f"\nTotal fall items: {total_fall_items}")
print(f"  - UR Fall videos: {ur_fall_info.get('fall_total', 0)}")
print(f"  - Le2i sequences: {le2i_info.get('fall_total', 0)}")
print(f"\nTotal ADL items: {total_adl_items}")
print(f"  - UR Fall videos: {ur_fall_info.get('adl_total', 0)}")
print(f"  - Le2i sequences: {le2i_info.get('adl_total', 0)}")
print(f"\nGrand total: {total_items}")

if total_items > 0:
    print(f"\nClass balance: {total_fall_items / total_items * 100:.1f}% falls, {total_adl_items / total_items * 100:.1f}% ADL")

if IN_COLAB:
    print(f"\n✓ Datasets saved in Google Drive at: {paths['data_raw']}")
    print(f"  (MyDrive/{GOOGLE_DRIVE_PROJECT_FOLDER}/data/raw/)")
else:
    print(f"\n✓ Datasets saved locally at: {paths['data_raw']}")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("1. Review downloaded data in 01_eda.ipynb")
print("2. Extract pose keypoints using 02_pose_extraction.ipynb")
print("3. Train models using 03_model_experiments.ipynb")
print("\nNote: If datasets need to be downloaded manually:")
print("  - UR Fall: http://fenix.ur.edu.pl/~mkepski/ds/uf.html")
print("  - Le2i: https://imvia.u-bourgogne.fr/en/database/fall-detection-dataset-2.html")